IGNORE cleaner_v2 muna. it runs better when they're in the same file.

In [19]:
import re

def divide_into_verses(text):
    # WEIRD ENCODING: replace weird ‘ with ' accounts for letters before and after
    text = re.sub(r"(\w)‘(\w)", r"\1'\2", text, flags=re.MULTILINE)
    text = re.sub(r"‘", r"'", text, flags=re.MULTILINE)  # for standalone
    # WEIRD ENCODING: replace ’ with '
    text = re.sub(r"(\w)’(\w)", r"\1'\2", text, flags=re.MULTILINE)
    text = re.sub(r"’", r"'", text, flags=re.MULTILINE)  # for standalone
    # WEIRD ENCODING: replace weird ” and “ with " and accounts for letters before and after
    text = re.sub(r"(\w)”", r'\1"', text, flags=re.MULTILINE)
    text = re.sub(r"“(\w)", r'"\1', text, flags=re.MULTILINE)
    text = re.sub(r"”", r'"', text, flags=re.MULTILINE)  # for standalone
    text = re.sub(r"“", r'"', text, flags=re.MULTILINE)  # for standalone

    # SPACING: remove double spaces
    text = re.sub(r' {2,}', ' ', text, flags=re.MULTILINE)

    # SPACING: remove double new lines
    #text = re.sub(r'\n\n', r'', text, flags = re.MULTILINE)

    # SPACING: remove leading newlines
    #text = re.sub(r'^\n', r'', text, flags = re.MULTILINE)

    # SPACING: remove leading whitespace (e.g., " Text" -> "Text")
    text = re.sub(r'^\s+', r'', text, flags=re.MULTILINE)

    text = re.sub(r'\d+ (\d+)', r'\1', text, flags = re.MULTILINE)
    text = re.sub(r'(\s)(\d+)([A-z])', r'\1\n\2\3', text, flags = re.MULTILINE)
    text = re.sub(r'^[MJL].*\n', r'', text, flags = re.MULTILINE)
    text = re.sub(r'(\d+)([A-z])', r'\1,\2', text, flags = re.MULTILINE)
    text = re.sub(r'(\d+) ([A-z])', r'\1,\2', text, flags = re.MULTILINE)
    text = re.sub(r'(\.)\s([0-9],[A-z])', r'\1\n\2', text, flags = re.MULTILINE)


    text = re.sub(r'', r'', text, flags = re.MULTILINE)
    #text = re.sub(r"(\d+)", r"\n", text, flags = re.MULTILINE)
    # ----- SPECIFIC QUIRKS BELOW -----
    text = re.sub(r'¶', r',', text, flags = re.MULTILINE)

    return text

In [20]:
def divide_into_sentences(text):
    # remove any numbers with spaces at the start of lines
    # example: "1 This is a verse -> "This is a verse"
    text = re.sub(r'^\d+\s*', r'', text, flags=re.MULTILINE)
    
    # removes book and chapter headings (e.g., "Matthew 1")
    # ^[A-Za-zÀ-ÖØ-öø-ÿ\s]+ matches the book name (also covers accented letters for other languaes that do have them)
    # \s+ matches the space between the book name and chapter number
    # \d+ matches the chapter number
    # \s*\n? matches any trailing spaces and newline
    text = re.sub(r'^[0-9A-Za-zÀ-ÖØ-öø-ÿ\s]+\d+\s*\n?', '', text, flags=re.MULTILINE)

    # removes verse numbers
    # (?![\d,\.]) makes sure we don't remove numbers that are part of decimals or commas
    text = re.sub(r'\b\d+(?![\d,\.])', '', text)

    # replaces multiple newlines with a single newline
    text = re.sub(r'\n+', r'\n', text)
    
    # replaces curly quotes with standard quotes
    text = re.sub(r"[“”]", '"', text)

    # replaces curly apostrophes with standard apostrophes
    text = re.sub(r"[‘’]", "'", text)

    # replaces multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    # removes spaces before punctuation (e.g., "Hello !" -> "Hello!")
    text = re.sub(r'\s+([.!?])', r'\1', text)

    # removes spaces after opening punctuation
    text = re.sub(r'([(\["“‘])\s+', r'\1', text)

    # removes spaces before closing punctuation
    text = re.sub(r'\s+([)\]”’"])', r'\1', text)

    # ensures there is a space after sentence-ending punctuation if followed by a non-space character
    # [.!?] matches sentence ending punctuation
    # ["'\)\]]? matches any closing quotes, parentheses, or brackets that may follow the punctuation
    # (?=\S) ensures the punctuation is followed by a non-space character
    text = re.sub(r'([.!?]["\')\]]?)(?=\S)', r'\1 ', text)

    # ensures each sentence starts on a new line
    # [.!?] matches sentence ending punctuation
    # ["'\)\]]* matches any closing quotes, parentheses, or brackets that may follow the punctuation
    # \s+ matches the whitespace following the punctuation
    text = re.sub(r'([.!?](?:["\'\)\]\}]+)?)\s+', r'\1\n', text)

    # removes leading whitespace (e.g., " Text" -> "Text")
    text = re.sub(r'^\s+', r'', text, flags=re.MULTILINE)

    # This regex keeps letters (including accented), numbers, and common punctuation marks 
    # while removing inconsistent special characters.
    # Basically, if its not in the list of characters, it gets omitted.
    # I had to consult AI with this one in giving me the characters to be included.
    text = re.sub(r'[^A-Za-zÀ-ÖØ-öø-ÿ\u00C0-\u024F\u1E00-\u1EFF0-9\s\.\,\!\?\:\;\'"\-\(\)]', "", text)

    return text

In [21]:
from pathlib import Path
# from cleaner_v2 import divide_into_verses # cleaner_v2.py

languages = ["spanish", "tagalog", "english", "hiligaynon", "bikol", "waray", "ilocano", "cebuano", "kapampangan", "pangasinense", "yakan", "ivatan", "tausug", "yami", "tuwali_ifugao", "masbateno"]
data_folder = Path("data")
cleaned_folder = Path("data/cleaned")
cleaned_folder.mkdir(parents=True, exist_ok=True) 

ALWAYS delete the cleaned folder before clicking Run All. tnx

In [22]:
for lang in languages:
    file_path = data_folder / f"{lang}.txt"
    output_path = cleaned_folder / f"{lang}-cleaned.txt"

    # SKIP if the input file does not exist or is empty
    if not file_path.exists() or file_path.stat().st_size == 0:
        print(f"Skipped: {file_path} (file does not exist or is empty)")
        continue

    # read input
    with file_path.open("r", errors="ignore", encoding="utf-8") as f:
        text = f.read()

    # clean the text
    cleaned_text = divide_into_verses(text)

    # save
    with output_path.open("w", encoding="utf-8") as f:
        f.write(cleaned_text)

    print(f"Cleaned and saved: {output_path}")

Cleaned and saved: data\cleaned\spanish-cleaned.txt
Cleaned and saved: data\cleaned\tagalog-cleaned.txt
Cleaned and saved: data\cleaned\english-cleaned.txt
Cleaned and saved: data\cleaned\hiligaynon-cleaned.txt
Cleaned and saved: data\cleaned\bikol-cleaned.txt
Cleaned and saved: data\cleaned\waray-cleaned.txt
Cleaned and saved: data\cleaned\ilocano-cleaned.txt
Cleaned and saved: data\cleaned\cebuano-cleaned.txt
Cleaned and saved: data\cleaned\kapampangan-cleaned.txt
Cleaned and saved: data\cleaned\pangasinense-cleaned.txt
Cleaned and saved: data\cleaned\yakan-cleaned.txt
Cleaned and saved: data\cleaned\ivatan-cleaned.txt
Cleaned and saved: data\cleaned\tausug-cleaned.txt
Cleaned and saved: data\cleaned\yami-cleaned.txt
Cleaned and saved: data\cleaned\tuwali_ifugao-cleaned.txt
Cleaned and saved: data\cleaned\masbateno-cleaned.txt


In [23]:
for lang in languages:
    file_path = data_folder / f"{lang}.txt"
    output_path = cleaned_folder / f"(sentences)-{lang}-cleaned.txt"

    # SKIP if the input file does not exist or is empty
    if not file_path.exists() or file_path.stat().st_size == 0:
        print(f"Skipped: {file_path} (file does not exist or is empty)")
        continue

    # read input
    with file_path.open("r", errors="ignore", encoding="utf-8") as f:
        text = f.read()

    # clean the text
    cleaned_text = divide_into_sentences(text)

    # save
    with output_path.open("w", encoding="utf-8") as f:
        f.write(cleaned_text)

    print(f"Cleaned and saved: {output_path}")

Cleaned and saved: data\cleaned\(sentences)-spanish-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-tagalog-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-english-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-hiligaynon-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-bikol-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-waray-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-ilocano-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-cebuano-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-kapampangan-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-pangasinense-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-yakan-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-ivatan-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-tausug-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-yami-cleaned.txt
Cleaned and saved: data\cleaned\(sentences)-tuwali_ifugao-cleaned.txt
Cleaned and saved: data\cleaned\(

In [24]:
'''import re
sample_text = "80 Nagdakula an aki asin nagkosog sa espiritu. 3 Nag-erok siya sa kalangtadan sagkod. 24 Sa aldaw na magpamidbid siya sa Israel."
# if a number of 1-2 digits is detected, within a line, regardless of position (start, middle, etc) on the line
# add a newline BEFORE it
sample_text = re.sub(r'(?<!\S)(\d{1,2}\b)', r'\n\1', sample_text)
sample_text'''

<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:5: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Van Asher Alcantara\AppData\Local\Temp\ipykernel_15164\333425405.py:5: SyntaxWarning: invalid escape sequence '\S'
  sample_text = re.sub(r'(?<!\S)(\d{1,2}\b)', r'\n\1', sample_text)


'import re\nsample_text = "80 Nagdakula an aki asin nagkosog sa espiritu. 3 Nag-erok siya sa kalangtadan sagkod. 24 Sa aldaw na magpamidbid siya sa Israel."\n# if a number of 1-2 digits is detected, within a line, regardless of position (start, middle, etc) on the line\n# add a newline BEFORE it\nsample_text = re.sub(r\'(?<!\\S)(\\d{1,2}\x08)\', r\'\n\x01\', sample_text)\nsample_text'